# Picker latency — v1.1 one-shot vs v2.0 resident tray app

- **File:** picker_latency.ipynb
- **Description:** Measured "link click → picker on screen" latency of Browser Picker Pro before and after the v2.0 resident/launcher redesign, step by step, with the measuring code and the analysis.
- **Author:** Jose-Jorge HERNANDEZ
- **Company:** Parlee Conseiller, Inc.
- **Date:** 2026-11-25
- **Last edit date:** 2026-11-25
- **Version:** 1.0.0

## Context
Goal: the picker is an *eye-candy* app whose background video should appear **immediately** when a link is clicked, and the app should stay resident (tray) instead of exiting after each pick.

Machine: a Windows 11 workstation, two 1920×1080 monitors, picker configured with a custom MP4 video background, Modal window, WrapGrid, 4 browsers (Chrome, Edge, Opera, Zen).

## Method

Two independent measurements:

1. **External** — `scripts/measure-picker-latency.ps1` starts the exe Windows would run (v1: `BrowserPickerPro.exe`; v2: `BrowserPickerProLauncher.exe`) with a URL via `Start-Process`, then polls every 2 ms for a visible, *on-screen* window titled "Browser Picker Pro". It includes PowerShell's own process-start overhead, so it over-states the real latency a little.
2. **Internal (v2 only)** — the launcher sends its own start time (`Stopwatch.GetTimestamp()`, machine-wide QPC ticks) through the pipe; the resident app logs `Picker first frame N ms after link click` on the first `CompositionTarget.Rendering` after `Reveal()`, plus a Debug breakdown (`launcher→host`, `settings+rules`, `prepare`, `show`).

The core of the external measurement (from the script) is below. It is PowerShell, so it runs here through a `%%script pwsh` cell on the python3 kernel; the data cells further down are plain Python.

In [ ]:
%%script pwsh -NoProfile -Command -
# Excerpt of scripts/measure-picker-latency.ps1 — the timed part of one run.
# (Running this cell needs a published build; the numbers below were captured on 2026-11-25.)
$Exe = 'H:\DEV\browser-picker-pro\publish\win-x64\BrowserPickerProLauncher.exe'
Add-Type -Namespace Bpp -Name Win -MemberDefinition @'
[DllImport("user32.dll", CharSet = CharSet.Unicode)] public static extern IntPtr FindWindowW(string cls, string title);
[DllImport("user32.dll")] public static extern bool IsWindowVisible(IntPtr h);
[StructLayout(LayoutKind.Sequential)] public struct RECT { public int Left, Top, Right, Bottom; }
[DllImport("user32.dll")] public static extern bool GetWindowRect(IntPtr h, out RECT r);
'@
$sw = [Diagnostics.Stopwatch]::StartNew()
Start-Process -FilePath $Exe -ArgumentList '"https://example.com/latency-test"'
while ($sw.ElapsedMilliseconds -lt 15000) {
    # [NullString]::Value — PowerShell would pass $null as "" and match nothing.
    $h = [Bpp.Win]::FindWindowW([NullString]::Value, 'Browser Picker Pro')
    if ($h -ne 0 -and [Bpp.Win]::IsWindowVisible($h)) {
        $r = New-Object Bpp.Win+RECT; [void][Bpp.Win]::GetWindowRect($h, [ref]$r)
        if ($r.Left -gt -30000) { break }   # v2 parks the window at -32000 between links
    }
    Start-Sleep -Milliseconds 2
}
"on screen after $($sw.ElapsedMilliseconds) ms"

## Results

Raw numbers captured during the 2026-11-25 session, in the order the optimisations were made:

| Stage | What changed |
|---|---|
| `v1.1` | Baseline: new 147 MB process per link, browser detection on every link |
| `v2-a` | Resident app + NativeAOT launcher; window Hide()/Show(); settings applied fully per link |
| `v2-b` | Settings applied only when changed; card list not rebuilt when unchanged |
| `v2-c` | Window parked off-screen instead of Hide()/Show() |
| `v2-d` | Pre-warm no longer waits for the video to open (fixes cold path) |

In [ ]:
# Measured data (ms). 'external' = measure-picker-latency.ps1, 'internal' = app log first frame.
from statistics import median

runs = {
    'v1.1': {'external': [1542, 1534, 1528, 1518, 1418, 1370], 'internal': []},
    'v2-a': {'external': [485, 308, 275, 117, 162, 162, 152, 106, 120, 119],
             'internal': [220.2, 109.2, 169.9, 124.0, 117.5, 133.1, 110.0, 111.5, 91.3, 111.0]},
    'v2-b': {'external': [229, 274, 217, 129, 136, 137, 109, 136, 201, 122],
             'internal': [90.7, 109.9, 82.9, 102.2]},
    'v2-c': {'external': [168, 110, 133, 102, 147, 206, 92, 91, 135, 280],
             'internal': [72.4, 79.4, 63.0]},
    'v2-d': {'external': [85, 77, 61, 61, 60],
             'internal': [51.6, 54.2, 50.5, 53.8, 56.3]},
}

print(f"{'stage':6} {'ext median':>10} {'ext min':>8} {'int median':>10} {'speed-up':>9}")
base = median(runs['v1.1']['external'])
for stage, d in runs.items():
    ext = median(d['external'])
    internal = f"{median(d['internal']):.0f}" if d['internal'] else '-'
    print(f"{stage:6} {ext:>10.0f} {min(d['external']):>8} {internal:>10} {base/ext:>8.1f}x")

Expected output of the cell above (computed from the same data):

| stage | external median | external min | internal median | speed-up vs v1.1 |
|---|---|---|---|---|
| v1.1 | 1523 | 1370 | – | 1.0× |
| v2-a | 157 | 106 | 114 | 9.7× |
| v2-b | 136 | 109 | 96 | 11.2× |
| v2-c | 134 | 91 | 72 | 11.4× |
| v2-d | 61 | 60 | 54 | 25.0× |

(`statistics.median` averages the two middle values for even counts, so v1.1 is 1523 here while the script's integer median printed 1518.)

### Where the remaining time goes (v2, Debug breakdown)

| Part | Before parking (v2-b) | After parking (v2-c) |
|---|---|---|
| launcher start + pipe hop | 21–44 ms | 12–22 ms |
| settings.json + rules read | 3–9 ms | 2–9 ms |
| view-model reset | 0.4 ms | 0.4–0.6 ms |
| window show/move + activate + video Play + animation start | 51–59 ms | 37–40 ms |

### Other paths

- **Cold start** (no resident instance, launcher starts the full app): internal first frame **1395 ms** (v2-d; was 3389 ms in v2-c because pre-warm waited up to 3 s for the video). External poll reported 2497 ms — the two disagree and the external one includes PowerShell overhead plus window-position polling; treat cold start as "roughly v1 speed". It happens once per sign-in, never with *Start with Windows* on.
- **Second `BrowserPickerPro.exe` forwarding** (main exe run directly while resident): 450 ms — it still boots .NET before forwarding, which is exactly why the launcher exists.
- **Launcher alone** (process start → exit, no main exe present): ~38 ms wall time from PowerShell, most of it `Start-Process` overhead.
- **Memory**: resident working set ~241–246 MB with the MP4 background open (≈91 MB before the video pipeline was involved). Deliberately not trimmed — see technical memory.

## Conclusion

- Click → first picker frame went from **~1.5 s to ~50–60 ms** (≈25× faster), under the ~100 ms threshold where a UI response is perceived as instant. The background video is already playing on that first frame because its decoder stays open while parked.
- The biggest single win is not running a fresh .NET process per link (v1.1 → v2-a). After that, the remaining cost was WPF work that must not be repeated per link: re-creating `ItemsPanel`/card containers and `Show()` of a layered window.
- Caveats: few samples per stage, measured on one machine while other work was running; v2-c → v2-d improvement is partly noise (v2-d's code change only affected the cold path). Screenshots were not usable for timing: a full-screen GDI capture stalls DWM composition by ~150 ms.